# API 调用基础

预计阅读时间：10-15 分钟。

大模型 API 调用本质上是一次 HTTP 请求。程序把 `model`、`messages` 和生成参数发送给服务端，服务端返回模型输出。

## OpenAI 兼容 API

很多大模型平台都提供 OpenAI 兼容 API。它的好处是：如果代码已经基于 OpenAI SDK 编写，迁移到其他兼容平台时，通常只需要修改两处：

- `api_key`：换成目标平台的 API Key；
- `base_url`：换成目标平台的 API 地址。

本课程采用智谱大语言模型，其 OpenAI 兼容 API 地址为：

In [4]:
BASE_URL = "https://open.bigmodel.cn/api/paas/v4/"
BASE_URL

'https://open.bigmodel.cn/api/paas/v4/'

智谱 GLM 的注册地址为：[https://bigmodel.cn/](https://bigmodel.cn/)。一般情况下，新用户注册后会获得一定数量的免费 tokens，可用于完成本章练习。

## 安装依赖

本节使用 `openai` SDK 调用 GLM 的 OpenAI 兼容接口。为了从 `.env` 文件读取环境变量，也安装 `python-dotenv`：

In [ ]:
!python3 -m pip install --upgrade "openai>=1.0" python-dotenv

## 保存 API Key

API Key 相当于密码，不要写进 notebook、代码文件或 Git 仓库。

### 方法一：在当前终端临时设置环境变量：
```bash
export ZAI_API_KEY="你的 API Key"
```


### 方法二：创建本地 `.env` 文件。
这个文件只放在自己机器上，不提交到 Git：
```bash
echo 'ZAI_API_KEY=你的 API Key' > .env
```

无论使用方法一还是方法二，Python 中都使用同一段代码读取 API Key：

In [6]:
import os
from dotenv import load_dotenv

# 如果当前目录存在 .env，load_dotenv 会把其中的变量加载到环境变量中。
# 如果已经在终端 export 过 ZAI_API_KEY，这行代码不会有副作用。
load_dotenv()

api_key = os.getenv("ZAI_API_KEY")
if not api_key:
    raise RuntimeError("请先设置环境变量 ZAI_API_KEY")

print("API Key 已读取")

API Key 已读取


## 创建客户端

`OpenAI` 客户端负责向服务端发送请求。这里的关键是把 `base_url` 指向 GLM 的 OpenAI 兼容接口：

In [7]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://open.bigmodel.cn/api/paas/v4/",
)

MODEL_NAME = "glm-5.2"  # 示例模型名，以平台当前可用模型为准

## 示例一：简单 prompt

先运行一个简单 prompt：

In [8]:
simple_prompt = "介绍 Transformer。"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "你是一个严谨、简洁的深度学习助教。"},
        {"role": "user", "content": simple_prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)

Transformer 是一种基于自注意力机制的深度学习架构，由 Vaswani 等于 2017 年提出。它摒弃了传统的 RNN/CNN 结构，完全依赖注意力机制处理序列数据。

**核心机制：自注意力**
通过计算序列中各元素间的点积相似度动态分配权重，捕获全局依赖。公式为：
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$
其中 $Q, K, V$ 分别为查询、键、值矩阵，$\sqrt{d_k}$ 用于缩放防止点积过大导致梯度消失。

**架构组件**
标准结构包含编码器和解码器，现代变体常仅使用其一（如 BERT 仅用编码器，GPT 仅用解码器）。主要模块包括：
1. **多头注意力**：将 Q, K, V 投影到多个子空间并行计算注意力，使模型能关注不同表征维度的信息。
2. **位置编码**：由于缺乏序列的归纳偏置，需显式注入位置信息（如正弦/余弦编码或可学习编码）。
3. **前馈神经网络**：对每个位置独立进行非线性变换（通常为两层线性映射加激活函数）。
4. **残差连接与层归一化**：缓解深层网络梯度消失，提升训练稳定性。

**核心优势**
* **高度并行**：摒弃时序依赖，序列内所有 token 可同时计算，大幅提升硬件利用率。
* **长距离依赖**：任意两个 token 间的计算路径长度为 $O(1)$，有效解决了 RNN 的长序列信息衰减问题。

Transformer 是当前自然语言处理（NLP）大模型的基石，并已成功扩展至计算机视觉（CV）、语音处理等多模态领域。


这个 prompt 通常能得到可读答案，但输出范围和格式不稳定。

## 示例二：清晰 prompt

再运行一个更清晰的 prompt：

In [9]:
clear_prompt = """请从模块层面介绍 Transformer，包括各个模块的输入输出维度。
采用 Markdown 格式返回，包括二级目录，不超过 200 字。"""

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "你是一个严谨、简洁的深度学习助教。"},
        {"role": "user", "content": clear_prompt},
    ],
    temperature=0.6,
)

print(response.choices[0].message.content)

设 $L$ 为序列长度，$d$ 为隐藏维度，$V$ 为词表大小。

## 输入模块
词嵌入与位置编码：输入 $(L)$，输出 $(L, d)$。

## 编码器模块
多头自注意力、前馈网络及残差归一化：输入 $(L, d)$，输出 $(L, d)$。

## 解码器模块
掩码自注意力、交叉注意力及前馈网络：输入 $(L, d)$，输出 $(L, d)$。

## 输出模块
线性层与Softmax：输入 $(L, d)$，输出 $(L, V)$。


比较两个输出时，重点看三点：

- 是否覆盖模块层面；
- 是否包含输入输出维度；
- 是否按 Markdown 和长度限制返回。

这说明 prompt 不是“装饰文字”，而是直接影响模型输出的接口。

## 请求体结构

上面的 SDK 调用最终会组织成类似下面的请求体：

In [10]:
request_body = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "你是一个严谨、简洁的深度学习助教。"},
        {"role": "user", "content": clear_prompt},
    ],
    "temperature": 0.6,
}

request_body

{'model': 'glm-5.2',
 'messages': [{'role': 'system', 'content': '你是一个严谨、简洁的深度学习助教。'},
  {'role': 'user',
   'content': '请从模块层面介绍 Transformer，包括各个模块的输入输出维度。\n采用 Markdown 格式返回，包括二级目录，不超过 200 字。'}],
 'temperature': 0.6}

## 小结

一次聊天 API 调用最核心的是：

- `api_key`：证明调用者身份；
- `base_url`：决定请求发到哪个平台；
- `model`：决定使用哪个模型；
- `messages`：提供 system 和 user 内容；
- `temperature` 等参数：影响生成的稳定性和随机性。

实际项目中，建议先写清楚 prompt，再用 API 调用验证输出是否稳定。

## 练习

把 `clear_prompt` 改成“介绍自注意力机制”，要求：

- 输出 Markdown；
- 包含 Query、Key、Value 的含义；
- 不超过 200 字。